# aiosignal: 5 Basic Usage Examples

Demonstrates the primary features of the [aiosignal](https://github.com/aio-libs/aiosignal) package. 

**aiosignal** is a list of registered asynchronous callbacks with a two-stage lifecycle:
1. **Building stage**: Use standard list operations (`append()`, `extend()`, `remove()`, etc.) to manage callbacks
2. **Frozen stage**: Call `freeze()` to lock the signal, then use `await signal.send(data)` to call all registered callbacks

This library is primarily used internally by aiohttp for managing application lifecycle events.


In [ ]:
# Example 1: Basic Signal creation and appending callbacks
from aiosignal import Signal

# Create a signal with an owner (required parameter)
owner = object()
sig = Signal(owner)

# Add async callbacks to the signal using standard list operations
async def callback1(value):
    print(f'Callback 1 received: {value}')

async def callback2(value):
    print(f'Callback 2 received: {value}')

# Add callbacks using list operations
sig.append(callback1)
sig.append(callback2)

print(f'Signal created with {len(sig)} callbacks')
print(f'Signal is frozen: {sig.frozen}')

# Before freezing, we can still modify the signal
print('Available methods:', [m for m in dir(sig) if not m.startswith('_') and callable(getattr(sig, m))])

# Demonstrate that we can add more callbacks before freezing
sig.extend([callback1])  # Can add same callback multiple times
print(f'After extend, signal has {len(sig)} callbacks')


In [ ]:
# Example 2: Freezing a signal and sending to callbacks
from aiosignal import Signal

# Create signal and add callbacks
sig = Signal(owner=object())
calls = []

async def handler1(data):
    calls.append(('h1', data))
    print(f'Handler 1 processed: {data}')

async def handler2(data):
    calls.append(('h2', data))
    print(f'Handler 2 processed: {data}')

# Add callbacks before freezing
sig.append(handler1)
sig.append(handler2)

# send() refuses to run until the signal is frozen
try:
    await sig.send('too early')
except RuntimeError as e:
    print(f'send() before freeze() raised: {e}')
else:
    raise AssertionError('send() on a non-frozen signal did not raise RuntimeError')

# Freeze the signal to prepare for sending
sig.freeze()
print(f'Signal is now frozen: {sig.frozen}')
assert sig.frozen

# After freezing, we can only send to callbacks, not modify the signal
async def demonstrate_sending():
    await sig.send('Hello from aiosignal!')
    await sig.send({'data': 'Complex object', 'count': 42})

await demonstrate_sending()
assert calls == [
    ('h1', 'Hello from aiosignal!'),
    ('h2', 'Hello from aiosignal!'),
    ('h1', {'data': 'Complex object', 'count': 42}),
    ('h2', {'data': 'Complex object', 'count': 42}),
], calls
print(f'{len(calls)} handler calls, in registration order')


In [ ]:
# Example 3: Managing callbacks with list operations
from aiosignal import Signal

sig = Signal(owner=object())
calls = []

async def handler1(arg):
    calls.append(f'Handler1: {arg}')
    print(f'Handler1: {arg}')

async def handler2(arg):
    calls.append(f'Handler2: {arg}')
    print(f'Handler2: {arg}')

async def handler3(arg):
    calls.append(f'Handler3: {arg}')
    print(f'Handler3: {arg}')

# Add multiple handlers using list operations
sig.extend([handler1, handler2, handler3])
print(f'Added {len(sig)} handlers')

# Remove a specific handler before freezing
sig.remove(handler2)
print(f'After removing handler2: {len(sig)} handlers remain')
assert list(sig) == [handler1, handler3]

# You can also use other list operations
sig.insert(1, handler2)  # Insert handler2 at position 1
print(f'After inserting handler2: {len(sig)} handlers')
assert list(sig) == [handler1, handler2, handler3]

# Freeze and send
sig.freeze()

async def send_to_all():
    await sig.send('Message to all handlers')

await send_to_all()
assert calls == [
    'Handler1: Message to all handlers',
    'Handler2: Message to all handlers',
    'Handler3: Message to all handlers',
], calls


In [ ]:
# Example 4: Practical async usage pattern (like in aiohttp)
from aiosignal import Signal

class SimpleServer:
    def __init__(self):
        self.on_startup = Signal(self)
        self.on_shutdown = Signal(self)
        self.events = []

    async def startup(self):
        """Called when server starts"""
        # Add startup callbacks
        self.on_startup.append(self.init_database)
        self.on_startup.append(self.init_cache)
        self.on_startup.freeze()

        print("Starting server...")
        await self.on_startup.send(self)

    async def shutdown(self):
        """Called when server stops"""
        # Add shutdown callbacks
        self.on_shutdown.append(self.close_database)
        self.on_shutdown.append(self.close_cache)
        self.on_shutdown.freeze()

        print("Shutting down server...")
        await self.on_shutdown.send(self)

    async def init_database(self, server):
        assert server is self
        self.events.append("init_database")
        print("Database initialized")

    async def init_cache(self, server):
        self.events.append("init_cache")
        print("Cache initialized")

    async def close_database(self, server):
        self.events.append("close_database")
        print("Database closed")

    async def close_cache(self, server):
        self.events.append("close_cache")
        print("Cache closed")

# Demonstrate the pattern
async def demo():
    server = SimpleServer()
    await server.startup()
    # ... server runs ...
    await server.shutdown()
    return server

server = await demo()
assert server.events == ["init_database", "init_cache", "close_database", "close_cache"], server.events


In [ ]:
# Example 5: Error handling and signal lifecycle
from aiosignal import Signal

sig = Signal(owner=object())
calls = []

async def good_handler(data):
    calls.append('good_handler')
    print(f'Good handler received: {data}')
    return f"processed: {data}"

async def error_handler(data):
    calls.append('error_handler')
    print(f'Error handler received: {data}')
    raise ValueError("Simulated error!")

async def another_handler(data):
    calls.append('another_handler')
    print(f'Another handler received: {data}')
    return f"also processed: {data}"

# Add handlers including one that will raise an error
sig.extend([good_handler, error_handler, another_handler])

# Freeze the signal
sig.freeze()

# Demonstrate error handling during send
async def demonstrate_error_handling():
    # send() awaits the receivers in order and stops at the first one that raises:
    # the exception reaches the caller and the receivers after it do not run.
    try:
        await sig.send("test data")
    except ValueError as e:
        print(f"Error caught: {e}")
    else:
        raise AssertionError("send() did not propagate the receiver's ValueError")
    assert calls == ['good_handler', 'error_handler'], calls  # another_handler never ran

    # Try to modify frozen signal (this will raise an error)
    try:
        sig.append(good_handler)
    except RuntimeError as e:
        print(f"Cannot modify frozen signal: {e}")
    else:
        raise AssertionError("append() on a frozen signal did not raise RuntimeError")

    # Show that signal is still functional after errors
    assert len(sig) == 3 and sig.frozen
    print(f"Signal still has {len(sig)} handlers and is frozen: {sig.frozen}")

await demonstrate_error_handling()
